# مُجسَّم Mujassam — تجميع النموذج الهجين (4.9B) وحفظه في Google Drive

هذا الدفتر يعمل على **Google Colab** (GPU مجاني T4 يكفي). ما يفعله:

1. يركّب Blender و TripoSR والمكتبات اللازمة.
2. يحمّل أوزان النموذج الهجين: **Qwen3-VL-4B** (العقل البصري: تخطيط، رؤية، نقد ذاتي) + **TripoSR** (تحويل الصورة إلى مجسم) + **U²-Net** (إزالة الخلفية) ≈ **4.9 مليار معامل**.
3. يضغط العقل إلى 4-bit (NF4) مع إبقاء مُرمِّز الرؤية بدقة 16-bit ليحافظ على جودة الرؤية.
4. يجرّب التوليد من نص ومن صورة.
5. **يحفظ الحزمة الكاملة في Google Drive** (`MyDrive/Mujassam/...`, حوالي 3.5–4 GB).
6. (اختياري) يدرّب النموذج تدريباً تخصصياً (LoRA) على بيانات Blender الاصطناعية ويحفظ المحوّل في Drive.
7. (اختياري) يشغّل خادم API ليتصل به تطبيقك على الهاتف.

> من القائمة: **Runtime → Change runtime type → T4 GPU** ثم شغّل الخلايا بالترتيب.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1) الإعدادات

In [ ]:
GITHUB_REPO = "alixer7/Game.html"          # المستودع الذي يحتوي مجلد mujassam
BRANCH      = "ccr-68964b11-loujs0"         # الفرع
PROFILE     = "mujassam-4.9b"               # أو mujassam-lite-2.6b لكروت أضعف
DRIVE_DIR   = "/content/drive/MyDrive/Mujassam"
PACKAGE_DIR = f"{DRIVE_DIR}/{PROFILE}"
BLENDER_VERSION = "4.2.3"                   # Blender 4.2 LTS

## 2) ربط Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs(DRIVE_DIR, exist_ok=True); print("Drive OK:", DRIVE_DIR)

## 3) جلب الكود
إذا كان المستودع خاصاً (private) سيُطلب منك GitHub token بصلاحية قراءة (لا يُحفظ).

In [ ]:
import os, subprocess, getpass
if not os.path.exists('/content/repo'):
    url = f"https://github.com/{GITHUB_REPO}.git"
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, "/content/repo"], capture_output=True, text=True)
    if r.returncode != 0:
        token = getpass.getpass("GitHub token (للمستودع الخاص): ")
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                        f"https://{token}@github.com/{GITHUB_REPO}.git", "/content/repo"], check=True)
%cd /content/repo/mujassam
!ls

## 4) تثبيت المكتبات + Blender + TripoSR (حوالي 5–8 دقائق)

In [ ]:
%%bash
set -e
pip install -q "transformers>=4.57.1" "accelerate>=1.0" "bitsandbytes>=0.45" "peft>=0.13" \
    rembg onnxruntime omegaconf einops trimesh huggingface_hub pillow \
    fastapi uvicorn python-multipart
pip install -q -e /content/repo/mujassam
# TripoSR (MIT) + marching cubes
[ -d /content/TripoSR ] || git clone -q --depth 1 https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR
pip install -q git+https://github.com/tatsy/torchmcubes.git
# Blender LTS (headless)
apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 libxxf86vm1 libgl1 > /dev/null

In [ ]:
import os, subprocess
bv = BLENDER_VERSION
bdir = f"/content/blender-{bv}-linux-x64"
if not os.path.exists(bdir):
    major = ".".join(bv.split(".")[:2])
    subprocess.run(f"wget -q https://download.blender.org/release/Blender{major}/blender-{bv}-linux-x64.tar.xz -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz -C /content", shell=True, check=True)
os.environ["MUJASSAM_BLENDER"] = f"{bdir}/blender"
os.environ["TRIPOSR_PATH"] = "/content/TripoSR"
print(subprocess.run([os.environ["MUJASSAM_BLENDER"], "-b", "--version"], capture_output=True, text=True).stdout.splitlines()[0])

## 5) اختبار Blender بدون ذكاء اصطناعي (بناء مواصفة جاهزة)

In [ ]:
!python -m mujassam.cli --out /content/test_chair build examples/specs/chair.json
from IPython.display import Image, display
display(Image('/content/test_chair/grid.png', width=512))

## 6) تحميل النموذج الهجين والتحقق من عدد المعاملات

In [ ]:
import logging; logging.basicConfig(level=logging.INFO, format="%(message)s")
from mujassam import MujassamModel
if os.path.exists(f"{PACKAGE_DIR}/mujassam_config.json"):
    m = MujassamModel.from_pretrained(PACKAGE_DIR, triposr_repo="/content/TripoSR")   # من Drive
else:
    m = MujassamModel.from_pretrained(PROFILE, triposr_repo="/content/TripoSR")       # من Hugging Face
print(m.num_parameters())
print(m.check_budget()[1])

## 7) توليد من نص (عربي أو إنجليزي)

In [ ]:
res = m.generate_from_text("كرسي خشبي حديث بأرجل رفيعة ووسادة زرقاء", out_dir="/content/out_text", preset="balanced")
print("score:", res.score, "attempts:", len(res.attempts), "seconds:", round(res.seconds))
for a in res.attempts: print(a.index, a.source, round(a.score, 2), a.critique.issues[:2])
display(Image(res.files["preview"], width=512))
print(res.files)

## 8) توليد من صورة

In [ ]:
from google.colab import files
up = files.upload()                       # اختر صورة لجسم واحد
img_path = list(up.keys())[0]
res_img = m.generate_from_image(img_path, prompt="", out_dir="/content/out_image", preset="balanced")
print("score:", res_img.score, [a.source for a in res_img.attempts])
display(Image(res_img.files["preview"], width=512))

## 9) حفظ الحزمة الكاملة في Google Drive
تُحفظ أوزان العقل مضغوطة (4-bit) + TripoSR + U²-Net + الإعدادات وبطاقة النموذج.

In [ ]:
out = m.save_pretrained(PACKAGE_DIR)
!du -sh "{PACKAGE_DIR}"
!ls -la "{PACKAGE_DIR}"
# نسخة من الكود أيضاً
!cd /content/repo && zip -qr "{DRIVE_DIR}/mujassam_code.zip" mujassam -x "*/__pycache__/*"
print("saved to", out)

## 10) (اختياري) تدريب تخصصي LoRA — يجعل العقل خبيراً في لغة المجسمات ونقد عمله

1. توليد بيانات اصطناعية بـ Blender (كل نموذج يعطي ~5 أمثلة: نص→مجسم، صورة→مجسم، تحليل، توجيه، ونقد/إصلاح).
2. تدريب QLoRA على T4 (حوالي 1–3 ساعات لـ 1500 نموذج).
3. حفظ المحوّل في Drive ثم إعادة حفظ الحزمة.

In [ ]:
!python training/synth_data.py --out /content/synth --n 1500 --res 256 --samples 8
!wc -l /content/synth/train.jsonl

In [ ]:
del m; import gc, torch; gc.collect(); torch.cuda.empty_cache()     # حرّر الذاكرة للتدريب
!python training/sft_lora.py --data /content/synth/train.jsonl --out "{DRIVE_DIR}/lora-r1" --epochs 2

In [ ]:
m = MujassamModel.from_pretrained(PACKAGE_DIR, brain_lora=f"{DRIVE_DIR}/lora-r1", triposr_repo="/content/TripoSR")
res = m.generate_from_text("منزل ريفي صغير بسقف قرميد أحمر ونافذتين وباب خشبي", out_dir="/content/out_lora")
print(res.score); display(Image(res.files["preview"], width=512))
m.save_pretrained(PACKAGE_DIR)   # الحزمة الآن تحمل محوّل LoRA داخلها (brain_lora/)

### التحسين الذاتي (Expert iteration)
النموذج يولّد ← يرى ← ينتقد ← يصلح، ونحتفظ بأفضل أعماله كبيانات تدريب جديدة ثم نعيد التدريب.

In [ ]:
!python training/self_improve.py --model "{PACKAGE_DIR}" --prompts training/prompts_ar_en.txt --out /content/round1 --preset quality
!python training/sft_lora.py --data /content/synth/train.jsonl /content/round1/train.jsonl --out "{DRIVE_DIR}/lora-r2" --epochs 1

## 11) (اختياري) خادم API لتطبيق الهاتف
يعطيك رابطاً عاماً مؤقتاً (Cloudflare) ومفتاح API. من التطبيق: `POST /v1/text-to-3d` ثم حمّل `model.glb`.

In [ ]:
import os, threading, secrets, subprocess, re, time
os.environ["MUJASSAM_SERVER_AUTOSTART"] = "0"
os.environ["MUJASSAM_API_KEY"] = API_KEY = secrets.token_urlsafe(16)
import uvicorn
from mujassam.server import create_app
import mujassam.server as srv; srv.API_KEY = API_KEY
threading.Thread(target=uvicorn.run, args=(create_app(m),), kwargs={"host": "0.0.0.0", "port": 8000, "log_level": "warning"}, daemon=True).start()
if not os.path.exists("/content/cloudflared"):
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared", shell=True, check=True)
p = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(60):
    line = p.stdout.readline()
    mm = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if mm: url = mm.group(0); break
print("API URL :", url)
print("API KEY :", API_KEY)
print(f'curl -X POST {url}/v1/text-to-3d -H "Authorization: Bearer {API_KEY}" -H "Content-Type: application/json" -d \'{{"prompt":"كوب قهوة"}}\'')